In [36]:
# !pip install --user imblearn

  Using cached imblearn-0.0-py2.py3-none-any.whl (1.9 kB)
  Using cached imbalanced_learn-0.9.1-py3-none-any.whl (199 kB)


In [30]:
import pandas as pd
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import train_test_split

In [6]:
dataframe = pd.read_csv("../datasets/dataset.csv")
dataframe["output_2"].value_counts()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_45156\286414837.py:1: DtypeWarning: Columns (10,40,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv("../datasets/dataset.csv")


SENSITIVE_DATA_EXPOSURE    26558
SQL_INJECTION               6573
NORMAL                      6350
WEB_SCAN                    5654
brute_force                  702
XSS                          317
Broken_Authentication        300
Name: output_2, dtype: int64

#  Now we need to make SMOTE records for each class


In [7]:
dataframe.shape

(46454, 43)

In [11]:
# all_features = ['_source.data.protocol',
#                          '_source.data.id',
#                          '_source.rule.firedtimes',
#                          '_source.rule.mail',
#                          '_source.rule.level',
#                          '_source.rule.description',
#                          '_source.rule.groups',
#                          '_source.rule.pci_dss',
#                          '_source.rule.tsc',
#                          '_source.rule.nist_800_53',
#                          '_source.rule.gdpr',
#                          '_source.rule.mitre.id',
#                          '_source.rule.frequency',
#                          '_source.rule.hipaa',
#                          '_source.agent.description',
#                         '_source.rule.id',

#                         'output_1',
#                         'output_2' ]

# category_features= ['_source.data.protocol',
#                      '_source.data.id',
#                      '_source.rule.mail',
#                      '_source.rule.description',
#                      '_source.rule.groups',
#                      '_source.rule.pci_dss',
#                      '_source.rule.tsc',
#                      '_source.rule.nist_800_53',
#                      '_source.rule.gdpr',
#                      '_source.rule.mitre.id',
#                      '_source.rule.hipaa',
#                         '_source.rule.id',

#                      '_source.agent.description',
#                      '_source.rule.frequency']
    
# numerical_features=['_source.rule.firedtimes',
#                     '_source.rule.level']



all_features = ['_source.data.protocol',
                         '_source.data.id',
                         '_source.rule.firedtimes',
                         '_source.rule.mail',
                         '_source.rule.level',
                         '_source.rule.description',
                         '_source.rule.groups',
                         '_source.rule.pci_dss',
                         '_source.rule.tsc',
                         '_source.rule.nist_800_53',
                         '_source.rule.gdpr',
                         '_source.rule.mitre.id',
                         '_source.rule.frequency',
                         '_source.rule.hipaa',
                         '_source.agent.description',
                        '_source.rule.id',
                        'output_1',
                        'output_2' ]
category_features= ['_source.data.protocol',
                         '_source.data.id',
                         '_source.rule.mail',
                         '_source.rule.description',
                         '_source.rule.groups',
                         '_source.rule.pci_dss',
                         '_source.rule.tsc',
                         '_source.rule.nist_800_53',
                         '_source.rule.gdpr',
                         '_source.rule.mitre.id',
                         '_source.rule.hipaa',
                         '_source.agent.description',
                        '_source.rule.id',

                         '_source.rule.frequency']

numerical_features = ['_source.rule.firedtimes',
                      '_source.rule.level'
                     ]


In [32]:
def preprocessing(df):  

    #these category features if they didn't exist we will add them to the dataframe
    list_test_columns= list(df.columns)
    for i in range ( 0, len(category_features)):
        if(category_features[i] not in list_test_columns):
            df[category_features[i]] = " "
    for c in numerical_features:
        if(c not in list_test_columns):
            df[c] = 1
    # fill the columns that has None values with empty strings.
    for c in category_features:
        if(df[c].isnull().any()):
            df[c] = df[c].fillna(' ')
    # fill the columns rule level with rule.level = 3
    df["_source.rule.level"] = df["_source.rule.level"].fillna(1)
    df["_source.rule.firedtimes"] = df["_source.rule.firedtimes"].fillna(1)

    # fill mail values with 1 and 0 
    for index,row in df.iterrows():
        if(isinstance(row["_source.rule.id"],int)):
            df.at[index,'_source.rule.id'] = str(row['_source.rule.id'])
        for cat_fe in category_features:
            if(isinstance(row[cat_fe],list)):
                df.at[index,cat_fe] = str(row[cat_fe])
        if (row["_source.rule.mail"]==True):
            df.at[index,"_source.rule.mail"] = 1
        else :
            df.at[index,"_source.rule.mail"] = 0
    
    
    df["_source.agent.description"] = "APACHE_SERVER"
    df = df.fillna(' ')
    return df

In [13]:
for index,row in dataframe.iterrows():
    for item in category_features:
        dataframe.at[index,item] = str(row[item])
    for item in numerical_features:
        dataframe.at[index,item ] = int(row[item])
dataframe

,_index,_type,_id,_score,_source.agent.ip,_source.agent.name,_source.agent.id,_source.manager.name,_source.data.protocol,_source.data.srcip,...,_source.rule.mitre.id,_source.rule.mitre.tactic,_source.rule.info,_source.decoder.parent,_source.data.srcip2,output_2,output_1,_source.agent.description,_source.rule.gpg13,_source.previous_log
0,wazuh-alerts-4.x-2022.07.19,_doc,3rE9FoIBL_7AcdV6DJfU,2.785566,10.10.50.7,ubuntu,12,ubuntu-linux,GET,10.10.30.158,...,,,,NaN,NaN,NORMAL,NORMAL,nan,NaN,NaN
1,wazuh-alerts-4.x-2022.07.19,_doc,4LE9FoIBL_7AcdV6DJfU,2.785566,10.10.50.7,ubuntu,12,ubuntu-linux,GET,10.10.30.158,...,,,,NaN,NaN,NORMAL,NORMAL,nan,NaN,NaN
2,wazuh-alerts-4.x-2022.07.19,_doc,4rE9FoIBL_7AcdV6DJfU,2.785566,10.10.50.7,ubuntu,12,ubuntu-linux,GET,10.10.30.158,...,,,,NaN,NaN,NORMAL,NORMAL,nan,NaN,NaN
3,wazuh-alerts-4.x-2022.07.19,_doc,47E9FoIBL_7AcdV6DJfU,2.785566,10.10.50.7,ubuntu,12,ubuntu-linux,GET,10.10.30.158,...,,,,NaN,NaN,NORMAL,NORMAL,nan,NaN,NaN
4,wazuh-alerts-4.x-2022.07.19,_doc,57E9FoIBL_7AcdV6DJfV,2.785566,10.10.50.7,ubuntu,12,ubuntu-linux,GET,10.10.30.158,...,,,,NaN,NaN,NORMAL,NORMAL,nan,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46449,NaN,NaN,NaN,NaN,192.168.200.135,ubuntu,3,wazuh-manager,POST,::ffff:10.0.0.10,...,,,NaN,NaN,NaN,SQL_INJECTION,ATTACK,APACHE_SERVER,,
46450,NaN,NaN,NaN,NaN,192.168.200.135,ubuntu,3,wazuh-manager,POST,::ffff:192.168.200.46,...,,,NaN,NaN,NaN,NORMAL,NORMAL,APACHE_SERVER,,
46451,NaN,NaN,NaN,NaN,192.168.200.135,ubuntu,3,wazuh-manager,POST,::ffff:10.0.0.10,...,,,NaN,NaN,NaN,SQL_INJECTION,ATTACK,APACHE_SERVER,,
46452,NaN,NaN,NaN,NaN,192.168.200.135,ubuntu,3,wazuh-manager,POST,::ffff:10.0.0.10,...,,,NaN,NaN,NaN,SQL_INJECTION,ATTACK,APACHE_SERVER,,


In [14]:
dataframe = preprocessing(dataframe)
dataframe = dataframe[all_features]
X = dataframe.drop(["output_1","output_2"],axis=1)
y= dataframe["output_2"]
X_train,X_validation,y_train,y_validation = train_test_split(X,y,test_size=0.2,random_state=0,stratify=y)
smote=SMOTENC(categorical_features=[0,1,3,5,6,7,8,9,10,11,12,13,14,15],random_state=10,sampling_strategy={"brute_force":1250,"XSS":1250,"Broken_Authentication":1250})
x_res,y_res = smote.fit_resample(X_train,y_train)

In [15]:
y_res.value_counts()

SENSITIVE_DATA_EXPOSURE    21246
SQL_INJECTION               5258
NORMAL                      5080
WEB_SCAN                    4523
brute_force                 1250
XSS                         1250
Broken_Authentication       1250
Name: output_2, dtype: int64

In [16]:
X_validation["output_2"] = y_validation
X_validation = X_validation.reset_index(drop=True)


In [17]:
X_validation["output_1"] = " " 
for index,row in X_validation.iterrows():
    if(row["output_2"] != "NORMAL"):
        X_validation.at[index,"output_1"]="ATTACK"
    else:
        X_validation.at[index,"output_1"] = "NORMAL"


In [18]:
X_validation.to_csv("../datasets/testing/final_testing_dataset_no_history.csv",index=False)

In [19]:
y_validation.value_counts()

SENSITIVE_DATA_EXPOSURE    5312
SQL_INJECTION              1315
NORMAL                     1270
WEB_SCAN                   1131
brute_force                 140
XSS                          63
Broken_Authentication        60
Name: output_2, dtype: int64

In [20]:
y_res.value_counts()

SENSITIVE_DATA_EXPOSURE    21246
SQL_INJECTION               5258
NORMAL                      5080
WEB_SCAN                    4523
brute_force                 1250
XSS                         1250
Broken_Authentication       1250
Name: output_2, dtype: int64

In [21]:
dataset = pd.DataFrame(x_res)
dataset["output_2"] = y_res
dataset["output_2"].value_counts()

SENSITIVE_DATA_EXPOSURE    21246
SQL_INJECTION               5258
NORMAL                      5080
WEB_SCAN                    4523
brute_force                 1250
XSS                         1250
Broken_Authentication       1250
Name: output_2, dtype: int64

In [22]:
dataset["output_1"] = " " 
for index,row in dataset.iterrows():
    if(row["output_2"] != "NORMAL"):
        dataset.at[index,"output_1"]="ATTACK"
    else:
        dataset.at[index,"output_1"] = "NORMAL"
        

In [23]:
dataset["output_1"].value_counts()

ATTACK    34777
NORMAL     5080
Name: output_1, dtype: int64

In [24]:
dataset_web_scan = dataset[dataset["output_2"]=="WEB_SCAN"]
dataset_web_scan["output_2"].value_counts()
dataset_sql_injection = dataset[dataset["output_2"]=="SQL_INJECTION"]
dataset_sql_injection["output_2"].value_counts()

SQL_INJECTION    5258
Name: output_2, dtype: int64

In [25]:
dataset_SENSITIVE_DATA_EXPOSURE = dataset[dataset["output_2"]=="SENSITIVE_DATA_EXPOSURE"]
dataset_SENSITIVE_DATA_EXPOSURE["output_2"].value_counts()

SENSITIVE_DATA_EXPOSURE    21246
Name: output_2, dtype: int64

In [26]:
dataset_web_scan_sample=dataset_web_scan.sample(1250,random_state=10)
dataset_SENSITIVE_DATA_EXPOSURE_sample=dataset_SENSITIVE_DATA_EXPOSURE.sample(1250,random_state=10)
dataset_sql_injection_sample=dataset_sql_injection.sample(1250,random_state=10)

In [27]:
final_dataset = dataset[dataset["output_2"] != "WEB_SCAN"]
final_dataset = final_dataset[final_dataset["output_2"] != "SENSITIVE_DATA_EXPOSURE"]
final_dataset = final_dataset[final_dataset["output_2"] != "SQL_INJECTION"]
final_dataset["output_2"].value_counts()

NORMAL                   5080
brute_force              1250
XSS                      1250
Broken_Authentication    1250
Name: output_2, dtype: int64

In [28]:
final_dataset = pd.concat([final_dataset,dataset_web_scan_sample,dataset_SENSITIVE_DATA_EXPOSURE_sample,dataset_sql_injection_sample],axis=0)
final_dataset["output_2"].value_counts()

NORMAL                     5080
brute_force                1250
XSS                        1250
Broken_Authentication      1250
WEB_SCAN                   1250
SENSITIVE_DATA_EXPOSURE    1250
SQL_INJECTION              1250
Name: output_2, dtype: int64

In [29]:
final_dataset.to_csv("../datasets/training/final_training_dataset_no_history_SMOTENC.csv",index=False)

# SMOTENC for the dataset with history

In [170]:
all_features= ['_source.data.protocol',
                         '_source.data.id',
                         '_source.rule.firedtimes',
                         '_source.rule.mail',
                         '_source.rule.level',
                         '_source.rule.description',
                         '_source.rule.groups',
                         '_source.rule.pci_dss',
                         '_source.rule.tsc',
                         '_source.rule.nist_800_53',
                         '_source.rule.gdpr',
                         '_source.rule.mitre.id',
                         '_source.rule.frequency',
                         '_source.rule.hipaa',
                         '_source.agent.description',
                         '_source.rule.id',
                         'history._source.rule.firedtimes',
                         'history._source.data.id.200',
                         'history._source.data.id.300',
                         'history._source.data.id.400',
                         'history._source.data.id.500',
                         'T1212',
                         'T1068',
                         'T1064',
                         'T1210',
                         'T1083',
                         'T1055',
                         'T1190',
                        'output_1',
                        'output_2' ]


category_features= ['_source.data.protocol',
                         '_source.data.id',
                         '_source.rule.mail',
                         '_source.rule.description',
                         '_source.rule.groups',
                         '_source.rule.pci_dss',
                         '_source.rule.tsc',
                         '_source.rule.nist_800_53',
                         '_source.rule.gdpr',
                         '_source.rule.mitre.id',
                         '_source.rule.hipaa',
                         '_source.agent.description',
                         '_source.rule.frequency',
                         '_source.rule.id']


numerical_features = ['_source.rule.firedtimes',
                      '_source.rule.level',
                      'history._source.rule.firedtimes',
                         'history._source.data.id.200',
                         'history._source.data.id.300',
                         'history._source.data.id.400',
                         'history._source.data.id.500',
                         'T1212',
                         'T1068',
                         'T1064',
                         'T1210',
                         'T1083',
                         'T1055',
                         'T1190']

In [171]:
dataframe = pd.read_csv("../datasets/dataset_with_history_7.csv")
dataframe["output_2"].value_counts()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_45156\3395141054.py:1: DtypeWarning: Columns (35,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv("../datasets/dataset_with_history_7.csv")


SENSITIVE_DATA_EXPOSURE    26558
SQL_INJECTION               6573
NORMAL                      6053
WEB_SCAN                    5654
brute_force                  702
XSS                          317
Broken_Authentication        300
Name: output_2, dtype: int64

In [172]:
dataframe["_source.agent.description"] = "APACHE_SERVER"

In [173]:
for index,row in dataframe.iterrows():
    for item in category_features:
        dataframe.at[index,item] = str(row[item])
    for item in numerical_features:
        dataframe.at[index,item ] = int(row[item])
dataframe

,_index,_type,_id,_score,_source.agent.ip,_source.agent.name,_source.agent.id,_source.manager.name,_source.data.protocol,_source.data.srcip,...,history._source.data.id.500,T1212,T1068,T1064,T1210,T1083,T1055,T1190,timestamp,count_events
0,NaN,NaN,NaN,NaN,192.168.200.135,ubuntu,3,wazuh-manager,POST,::ffff:10.0.0.10,...,0,0,0,0,0,0,0,0,2022-06-21 07:45:00,1
1,NaN,NaN,NaN,NaN,192.168.200.135,ubuntu,3,wazuh-manager,POST,::ffff:10.0.0.10,...,0,0,0,0,0,0,0,0,2022-06-21 07:53:00,1
2,NaN,NaN,NaN,NaN,192.168.200.135,ubuntu,3,wazuh-manager,POST,::ffff:10.0.0.10,...,0,0,0,0,0,0,0,0,2022-06-21 07:54:00,2
3,NaN,NaN,NaN,NaN,192.168.200.135,ubuntu,3,wazuh-manager,POST,::ffff:10.0.0.10,...,0,0,0,0,0,0,0,0,2022-06-21 07:57:00,3
4,NaN,NaN,NaN,NaN,192.168.200.135,ubuntu,3,wazuh-manager,POST,::ffff:10.0.0.10,...,0,0,0,0,0,0,0,0,2022-06-21 07:57:00,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46152,wazuh-alerts-4.x-2022.07.20,_doc,XPocG4IBUctoCOF1fO1t,3.701891,10.10.50.7,ubuntu,12,ubuntu-linux,GET,10.10.30.158,...,0,0,0,0,0,0,0,0,2022-07-20 10:17:00,7
46153,wazuh-alerts-4.x-2022.07.20,_doc,WvocG4IBUctoCOF1fO1t,3.717812,10.10.50.7,ubuntu,12,ubuntu-linux,GET,10.10.30.158,...,0,0,0,0,0,0,0,0,2022-07-20 10:17:00,7
46154,wazuh-alerts-4.x-2022.07.20,_doc,bRDRH4IB2lDw03ZsNqDu,3.717812,10.10.50.7,ubuntu,12,ubuntu-linux,GET,10.10.30.158,...,0,0,0,0,0,0,0,0,2022-07-20 10:17:00,7
46155,wazuh-alerts-4.x-2022.07.20,_doc,3BDRH4IB2lDw03ZsPKcT,3.701891,10.10.50.7,ubuntu,12,ubuntu-linux,PUT,10.10.30.158,...,0,0,0,0,0,0,0,0,2022-07-20 10:23:00,7


In [174]:
dataframe["_source.agent.description"].value_counts()

APACHE_SERVER    46157
Name: _source.agent.description, dtype: int64

In [175]:
dataframe = preprocessing(dataframe)
dataframe = dataframe[all_features]
X = dataframe.drop(["output_1","output_2"],axis=1)
y= dataframe["output_2"]
X_train,X_validation,y_train,y_validation = train_test_split(X,y,test_size=0.2,random_state=0,stratify=y)
smote=SMOTENC(categorical_features=[0,1,3,5,6,7,8,9,10,11,12,13,14,15],random_state=10,sampling_strategy={"brute_force":1250,"XSS":1250,"Broken_Authentication":1250})
x_res,y_res = smote.fit_resample(X_train,y_train)

In [176]:
X_validation["output_2"] = y_validation
X_validation = X_validation.reset_index(drop=True)
X_validation["output_1"] = " " 
for index,row in X_validation.iterrows():
    if(row["output_2"] != "NORMAL"):
        X_validation.at[index,"output_1"]="ATTACK"
    else:
        X_validation.at[index,"output_1"] = "NORMAL"
X_validation

,_source.data.protocol,_source.data.id,_source.rule.firedtimes,_source.rule.mail,_source.rule.level,_source.rule.description,_source.rule.groups,_source.rule.pci_dss,_source.rule.tsc,_source.rule.nist_800_53,...,history._source.data.id.500,T1212,T1068,T1064,T1210,T1083,T1055,T1190,output_2,output_1
0,GET,200,5483,0,1,Ignored URLs (simple queries).,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,0,NORMAL,NORMAL
1,GET,206,760,0,1,Ignored URLs (simple queries).,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,0,SENSITIVE_DATA_EXPOSURE,ATTACK
2,GET,200,254,0,1,Access log messages grouped.,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,3,XSS,ATTACK
3,GET,200,77,0,6,A web attack returned code 200 (success).,"[""web"",""accesslog"",""attack""]","[""6.5"",""11.4""]","[""CC6.6"",""CC7.1"",""CC8.1"",""CC6.1"",""CC6.8"",""CC7....","[""SA.11"",""SI.4""]",...,0,0,0,0,0,0,0,0,SENSITIVE_DATA_EXPOSURE,ATTACK
4,GET,200,288,0,1,Ignored URLs (simple queries).,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,0,SENSITIVE_DATA_EXPOSURE,ATTACK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9227,GET,500,1009,0,5,Web server 500 error code (Internal Error).,"[""web"",""accesslog"",""system_error""]",,,,...,7,0,0,0,0,3,3,3,WEB_SCAN,ATTACK
9228,GET,200,1860,0,1,Access log messages grouped.,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,0,SQL_INJECTION,ATTACK
9229,GET,200,2487,0,1,Ignored URLs (simple queries).,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,1,SENSITIVE_DATA_EXPOSURE,ATTACK
9230,GET,200,665,0,1,Access log messages grouped.,"[""web"",""accesslog""]",,,,...,2,0,0,0,0,1,1,2,SENSITIVE_DATA_EXPOSURE,ATTACK


In [177]:
X_validation.to_csv("../datasets/testing/final_testing_dataset_with_history_7.csv",index=False)

In [178]:
dataset = pd.DataFrame(x_res)
dataset["output_2"] = y_res
dataset["output_2"].value_counts()

SENSITIVE_DATA_EXPOSURE    21246
SQL_INJECTION               5258
NORMAL                      4842
WEB_SCAN                    4523
brute_force                 1250
XSS                         1250
Broken_Authentication       1250
Name: output_2, dtype: int64

In [179]:
dataset["output_1"] = " " 
for index,row in dataset.iterrows():
    if(row["output_2"] != "NORMAL"):
        dataset.at[index,"output_1"]="ATTACK"
    else:
        dataset.at[index,"output_1"] = "NORMAL"
  

In [180]:
dataset_web_scan = dataset[dataset["output_2"]=="WEB_SCAN"]
dataset_web_scan["output_2"].value_counts()
dataset_sql_injection = dataset[dataset["output_2"]=="SQL_INJECTION"]
dataset_sql_injection["output_2"].value_counts()

SQL_INJECTION    5258
Name: output_2, dtype: int64

In [181]:
dataset_SENSITIVE_DATA_EXPOSURE = dataset[dataset["output_2"]=="SENSITIVE_DATA_EXPOSURE"]
dataset_SENSITIVE_DATA_EXPOSURE["output_2"].value_counts()

SENSITIVE_DATA_EXPOSURE    21246
Name: output_2, dtype: int64

In [182]:
dataset_web_scan_sample=dataset_web_scan.sample(1250,random_state=10)
dataset_SENSITIVE_DATA_EXPOSURE_sample=dataset_SENSITIVE_DATA_EXPOSURE.sample(1250,random_state=10)
dataset_sql_injection_sample=dataset_sql_injection.sample(1250,random_state=10)

In [183]:
final_dataset = dataset[dataset["output_2"] != "WEB_SCAN"]
final_dataset = final_dataset[final_dataset["output_2"] != "SENSITIVE_DATA_EXPOSURE"]
final_dataset = final_dataset[final_dataset["output_2"] != "SQL_INJECTION"]
final_dataset["output_2"].value_counts()

NORMAL                   4842
brute_force              1250
XSS                      1250
Broken_Authentication    1250
Name: output_2, dtype: int64

In [184]:
final_dataset = pd.concat([final_dataset,dataset_web_scan_sample,dataset_SENSITIVE_DATA_EXPOSURE_sample,dataset_sql_injection_sample],axis=0)
final_dataset["output_2"].value_counts()

NORMAL                     4842
brute_force                1250
XSS                        1250
Broken_Authentication      1250
WEB_SCAN                   1250
SENSITIVE_DATA_EXPOSURE    1250
SQL_INJECTION              1250
Name: output_2, dtype: int64

In [185]:
final_dataset.shape

(12342, 30)

In [186]:
final_dataset.to_csv("../datasets/training/final_training_dataset_with_history_7_SMOTENC.csv",index=False)

In [28]:
final_dataset.columns

Index(['_source.data.protocol', '_source.data.id', '_source.rule.firedtimes',
       '_source.rule.mail', '_source.rule.level', '_source.rule.description',
       '_source.rule.groups', '_source.rule.pci_dss', '_source.rule.tsc',
       '_source.rule.nist_800_53', '_source.rule.gdpr',
       '_source.rule.mitre.id', '_source.rule.frequency', '_source.rule.hipaa',
       '_source.agent.description', '_source.rule.id',
       'history._source.rule.firedtimes', 'history._source.data.id.200',
       'history._source.data.id.300', 'history._source.data.id.400',
       'history._source.data.id.500', 'T1212', 'T1068', 'T1064', 'T1210',
       'T1083', 'T1055', 'T1190', 'output_2', 'output_1'],
      dtype='object')

In [157]:
tt = pd.read_csv("../datasets/dataset.csv")
tt["_source.data.url"].value_counts()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_7088\3312498266.py:1: DtypeWarning: Columns (10,40,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  tt = pd.read_csv("../datasets/dataset.csv")


/rest/products/search?q=                                                                    6014
/rest/admin/application-configuration                                                       1292
/rest/user/login                                                                            1241
/rest/user/whoami                                                                            648
/api/Challenges/                                                                             439
                                                                                            ... 
/rest/admin/gawwdrt02i6k                                                                       1
/rest/products/search?q=%29%29%20AND%204309%3D4309%20AND%20%28%282984%3D2984                   1
/rest/products/search?q=%27%29%29%20AND%205684%3D9734%20AND%20%28%28%27IjQH%27%3D%27IjQH       1
/api/;<ScRiPt/acu%20src=//xss.bxss.me/t/xss.js%3F9102></ScRiPt>                                1
/rest/products/search?q=banana